In [1]:
!pip install gcsfs pyarrow pandas h3 --quiet


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
# Cell 2 — Load January 2023 (1 bulan saja untuk prototype)
import pandas as pd
import gcsfs

fs = gcsfs.GCSFileSystem()

GCS_PATH = "hardy-geo-de-267342/raw/tlc_yellow/year=2023/month=01/yellow_tripdata_2023-01.parquet"

df = pd.read_parquet(f"gs://{GCS_PATH}", filesystem=fs)

print(f"Row count: {len(df):,}")
print(f"Columns: {list(df.columns)}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

Row count: 3,066,766
Columns: ['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'airport_fee', 'year', 'month']
Memory usage: 453.6 MB


In [3]:
# Cell 3 — Apply trip filters (D-040 Standard Clean)
row_before = len(df)

df_clean = df[
    # Null checks datetime
    df["tpep_pickup_datetime"].notna() &
    df["tpep_dropoff_datetime"].notna() &
    # Null checks location
    df["PULocationID"].notna() &
    df["DOLocationID"].notna() &
    # No time travel + max 3 hours
    (df["tpep_dropoff_datetime"] > df["tpep_pickup_datetime"]) &
    ((df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]).dt.total_seconds() <= 10800) &
    # Valid zones (exclude Unknown 264, 265)
    (~df["PULocationID"].isin([264, 265])) &
    (~df["DOLocationID"].isin([264, 265])) &
    # Passenger count (cast to int first)
    (df["passenger_count"].astype("Int64") >= 1) &
    (df["passenger_count"].astype("Int64") <= 8) &
    # Distance
    (df["trip_distance"] >= 0.1) &
    (df["trip_distance"] <= 100) &
    # Fare
    (df["fare_amount"] >= 2.50) &
    (df["fare_amount"] <= 500) &
    # Total amount
    (df["total_amount"] >= 2.50) &
    (df["total_amount"] <= 500)
]

row_after = len(df_clean)
dropped = row_before - row_after
pct_dropped = (dropped / row_before) * 100

print(f"Before : {row_before:,}")
print(f"After  : {row_after:,}")
print(f"Dropped: {dropped:,} ({pct_dropped:.1f}%)")

Before : 3,066,766
After  : 2,823,775
Dropped: 242,991 (7.9%)


In [4]:
# Cell 4 — Build lookup table: LocationID → h3_cell
import geopandas as gpd
import h3
import pandas as pd

# Load taxi zones
gdf_zones = gpd.read_file(
    "gs://hardy-geo-de-267342/raw/reference/nyc_taxi_zones.geojson",
    filesystem=fs
)

print(f"Zones loaded: {len(gdf_zones)} polygons")
print(f"CRS: {gdf_zones.crs}")
print(gdf_zones[["LocationID", "zone", "borough"]].head())

DataSourceError: No valid GCS credentials found. For authenticated requests, you need to set GS_SECRET_ACCESS_KEY, GS_ACCESS_KEY_ID, GS_OAUTH2_REFRESH_TOKEN, GOOGLE_APPLICATION_CREDENTIALS, or other configuration options, or create a /home/codespace/.boto file. Consult https://gdal.org/en/stable/user/virtual_file_systems.html#vsigs-google-cloud-storage-files for more details. For unauthenticated requests on public resources, set the GS_NO_SIGN_REQUEST configuration option to YES.